# Chunking and Indexing

Chunk the corpus with five strategies, index each in Qdrant and compare them.

In [1]:
from prime_land.config import CONFIG, get_path
from prime_land.ingest_document.chunkers import chunk_documents
from prime_land.ingest_document.comparison import compare_strategies
from prime_land.ingest_document.indexer import collection_name, get_client, index_chunks, save_chunks
from prime_land.ingest_document.writers import load_corpus
from prime_land.providers import get_embeddings

documents = load_corpus(get_path("corpus_file"), CONFIG["corpus"]["exclude_patterns"])
strategies = CONFIG["chunking"]["strategies"]
print(len(documents), "documents")

c:\Users\Sasiri_A\Documents\projects\python\prime_land\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


77 documents


In [2]:
chunk_sets = {}
for strategy in strategies:
    chunk_sets[strategy] = chunk_documents(documents, strategy)
    save_chunks(chunk_sets[strategy], strategy)
    print(strategy, len(chunk_sets[strategy]), "chunks")

semantic 377 chunks
fixed 161 chunks
sliding 298 chunks
parent_child 490 chunks
late 391 chunks


In [3]:
embeddings = get_embeddings()
try:
    client.close()
except NameError:
    pass
client = get_client()
stores = {}
for strategy in strategies:
    print("indexing", strategy)
    stores[strategy] = index_chunks(client, embeddings, chunk_sets[strategy], strategy, documents)
    print(strategy, client.count(collection_name(strategy)).count, "of", len(chunk_sets[strategy]))


Loading weights: 100%|██████████| 134/134 [00:02<00:00, 50.25it/s]


indexing semantic
semantic 832 of 377
indexing fixed
fixed 346 of 161
indexing sliding
sliding 661 of 298
indexing parent_child
parent_child 1070 of 490
indexing late
late 539 of 391


In [4]:
table = compare_strategies(client, stores, chunk_sets)
table

,strategy,chunk_count,avg_tokens,index_size_mb,avg_retrieval_ms
0,semantic,377,236.8,2.56,150.1
1,fixed,161,583.0,1.06,116.1
2,sliding,298,480.9,2.03,128.5
3,parent_child,490,203.3,3.29,114.6
4,late,391,248.5,1.66,102.4


In [5]:
client.close()